In [2]:
#### This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/alx-football-players/football_players.csv


In [35]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

The next 4 questions (Questions 35 - 38) are based on the FIFA players dataset (football_players.csv).

Please note that you may have some difficulties importing this file into pandas, and you may need to do some research to figure out how to do so successfully.

**What is the most common Overall score for players in the database?**

* 67
* 93
* 64
* 66

In [4]:
# Load the dataset
df = pd.read_csv('/kaggle/input/alx-football-players/football_players.csv',encoding='ISO-8859-1', index_col=0)
df.head()

,Age,Nationality,Overall,Acceleration,Aggression,Agility,Balance,Ball control,Composure,Crossing,...,Shot power,Sliding tackle,Sprint speed,Stamina,Standing tackle,Strength,Vision,Volleys,Preferred Positions,Preferred Positions Type
Name,,,,,,,,,,,,,,,,,,,,,
Cristiano Ronaldo,32,Portugal,94.0,89.0,63.0,89.0,63.0,93.0,95.0,85.0,...,94.0,23.0,91.0,92.0,31.0,80.0,85.0,88.0,ST LW,Forward
L. Messi,30,Argentina,93.0,92.0,48.0,90.0,95.0,95.0,96.0,77.0,...,85.0,26.0,87.0,73.0,28.0,59.0,90.0,85.0,RW,Forward
Neymar,25,Brazil,92.0,94.0,56.0,96.0,82.0,95.0,92.0,75.0,...,80.0,33.0,90.0,78.0,24.0,53.0,80.0,83.0,LW,Forward
L. Suárez,30,Uruguay,92.0,88.0,78.0,86.0,60.0,91.0,83.0,77.0,...,87.0,38.0,77.0,89.0,45.0,80.0,84.0,88.0,ST,Forward
M. Neuer,31,Germany,92.0,58.0,29.0,52.0,35.0,48.0,70.0,15.0,...,25.0,11.0,61.0,44.0,10.0,83.0,70.0,11.0,GK,GoalKeeper


In [6]:
# Check the column names to ensure 'Overall' is the correct column name
print(df.info)

<bound method DataFrame.info of                    Age Nationality  Overall  Acceleration  Aggression  \
Name                                                                    
Cristiano Ronaldo   32    Portugal     94.0          89.0        63.0   
L. Messi            30   Argentina     93.0          92.0        48.0   
Neymar              25      Brazil     92.0          94.0        56.0   
L. Suárez           30     Uruguay     92.0          88.0        78.0   
M. Neuer            31     Germany     92.0          58.0        29.0   
...                ...         ...      ...           ...         ...   
A. Kelsey           17     England     46.0          24.0        16.0   
B. Richardson       47     England     46.0          25.0        44.0   
J. Young            17    Scotland     46.0          66.0        26.0   
J. Lundstram        18     England     46.0          57.0        46.0   
L. Sackey           18       Ghana     46.0          48.0        52.0   

                  

In [8]:
# Calculate the most common Overall score
most_common_overall_score = df['Overall'].mode()[0]

print(f"The most common Overall score is: {most_common_overall_score}")

The most common Overall score is: 66.0


Construct a dataset that is a subset of players who can play in central defence (i.e. who have 'CB' somewhere in their Preferred Positions field).

Split this group into three:

World Class: overall score of 80 or more
Good: overall score of 70-79
Mediocre: overall score below 70
Now build a random forest classifier with default parameters apart from setting to 500 trees and setting the random seed to 1971, on ALL of the data for these central defenders, where the target variable is the classification into one of the three classes defined above, and the candidate features are all other numerical variables.

**In descending order, which are the five most important features that emerge from this model?**

Hint: Search sklearn random forest documentation for feature importance if you don’t have any idea how to establish this.

* Marking, Reactions, Composure, Standing Tackle, Sliding Tackle
* Standing tackle, Sliding Tackle, Composure, Marking, Heading accuracy
* Standing tackle, Marking, Interceptions, Sliding Tackle, Reactions
* Composure, Standing Tackle, Marking, Heading accuracy, Sliding Tackle
* Composure, Sliding Tackle, Aggression, Short passing, Marking

In [20]:
# Filter central defenders
central_defenders = df[df['Preferred Positions'].str.contains('CB')].copy()

# Define the target variable
def classify_overall(overall_score):
    if overall_score >= 80:
        return 'World Class'
    elif 70 <= overall_score < 80:
        return 'Good'
    else:
        return 'Mediocre'
    
    # Use .loc to avoid SettingWithCopyWarning
central_defenders.loc[:, 'Class'] = central_defenders['Overall'].apply(classify_overall)

In [21]:
# Define features and target
features = central_defenders.drop(columns=['Overall', 'Preferred Positions', 'Class'])
target = central_defenders['Class']

# Identify categorical columns
categorical_columns = features.select_dtypes(include=['object']).columns

# Create a ColumnTransformer to handle categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', features.columns.difference(categorical_columns)),
        ('cat', OneHotEncoder(), categorical_columns)
    ],
    remainder='passthrough'
)

# Preprocessing and training pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=500, random_state=1971))
])

In [22]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.25, random_state=1971)

# Fit the model
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', 'passthrough',
                                                  Index(['Acceleration', 'Age', 'Aggression', 'Agility', 'Balance',
       'Ball control', 'Composure', 'Crossing', 'Curve', 'Dribbling',
       'Finishing', 'Free kick accuracy', 'GK diving', 'GK handling',
       'GK kicking', 'GK positioning', 'GK reflexes', 'Heading accuracy',
       'Intercep...ots', 'Marking',
       'Penalties', 'Positioning', 'Reactions', 'Short passing', 'Shot power',
       'Sliding tackle', 'Sprint speed', 'Stamina', 'Standing tackle',
       'Strength', 'Vision', 'Volleys'],
      dtype='object')),
                                                 ('cat', OneHotEncoder(),
                                                  Index(['Nationality', 'Preferred Positions Type'], dtype='object'))])),
                ('classifier',
                 RandomForestClassifier(n_estimators=500, random_state=1971))])

In [31]:
# Get feature importances
importances = pipeline.named_steps['classifier'].feature_importances_

# Extract feature names after preprocessing
num_features = numerical_columns
cat_feature_names = pipeline.named_steps['preprocessor'].transformers_[1][1].get_feature_names_out()
feature_names = list(num_features) + list(cat_feature_names)

# Create DataFrame for feature importances
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

In [33]:
# Display the top 5 features
print("Top 5 features by importance:")
print(feature_importance_df.head())

Top 5 features by importance:
             Feature  Importance
31   Standing tackle    0.152552
22           Marking    0.117092
18     Interceptions    0.095335
28    Sliding tackle    0.091471
17  Heading accuracy    0.074426


Split the data into test and training sets, with 33% of the data reserved for the test set and a random seed of 911.

Compare k nearest neighbours (KNN) models with k varying from 1 to 5. **Which k gives rise to the best F1 score for the world class good groups respectively?**

* World class: k=4, Good: k=5
* World class: k=5, Good: k=4
* World class: k=1, Good: k=2
* World class: k=1, Good: k=4
* World class: k=5, Good: k=5

In [36]:
# Identify numerical columns
numerical_columns = features.select_dtypes(include=['number']).columns

# Create a ColumnTransformer to scale numerical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_columns)
    ]
)

In [40]:
# Initialize dictionaries to store F1 scores
f1_scores_world_class = []
f1_scores_good = []

# Test KNN with k from 1 to 5
for k in range(1, 6):
    # Create a pipeline with preprocessing and KNN classifier
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', KNeighborsClassifier(n_neighbors=k))
    ])

    # Fit the model
    pipeline.fit(X_train, y_train)

    # Predict on the test set
    y_pred = pipeline.predict(X_test)

    # Compute F1 scores for each class with multiclass averaging
    f1_world_class = f1_score(y_test, y_pred, labels=['World Class'], average='macro')
    f1_good = f1_score(y_test, y_pred, labels=['Good'], average='macro')

    f1_scores_world_class.append(f1_world_class)
    f1_scores_good.append(f1_good)

# Determine the best k for each class
best_k_world_class = f1_scores_world_class.index(max(f1_scores_world_class)) + 1
best_k_good = f1_scores_good.index(max(f1_scores_good)) + 1

print(f"Best k for World Class: {best_k_world_class}")
print(f"Best k for Good: {best_k_good}")

Best k for World Class: 1
Best k for Good: 5
